Install all the dependencies

In [3]:
!pip install langchain chromadb faiss-cpu sentence-transformers pandas langchain-community

  Using cached aiohttp-3.12.15-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.7 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.7.0-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
  Using cached multidict-6.6.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (5.3 kB)
  Using cached propcache-0.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached yarl-1.20.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (73 kB)
  Using cached marshmallow-3.26.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metada

Import all the installed dependencies

In [4]:
import pandas as pd
import numpy as np
from typing import List, Tuple

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document

Load Dataset

In [5]:
url = "https://raw.githubusercontent.com/Bluedata-Consulting/GAAPB01-training-code-base/refs/heads/main/Assignments/assignment2dataset.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (25, 3)


,course_id,title,description
0,C001,Foundations of Machine Learning,Understand foundational machine learning algor...
1,C002,Deep Learning with TensorFlow and Keras,Explore neural network architectures using Ten...
2,C003,Natural Language Processing Fundamentals,Dive into NLP techniques for processing and un...
3,C004,Computer Vision and Image Processing,Learn the principles of computer vision and im...
4,C005,Reinforcement Learning Basics,Get introduced to reinforcement learning parad...


Prepare Course Documents

In [6]:
# each course becomes a "document" with metadata
docs = []
for idx, row in df.iterrows():
    content = f"{row['title']} - {row['description']}"
    docs.append(Document(page_content=content, metadata={"course_id": row['course_id']}))
    
print("Prepared docs:", len(docs))

Prepared docs: 25


In [7]:
# use HuggingFace model for semantic embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# build FAISS index
vectorstore = FAISS.from_documents(docs, embedding_model)

print("Vector DB built with", len(docs), "courses.")

/tmp/ipykernel_6760/4018360642.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/home/zadmin/Desktop/assignments/GenAI_GCP_2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector DB built with 25 courses.


In [8]:
# recommentation function for courses
def recommend_courses(profile: str, completed_ids: List[str], k: int = 5) -> List[Tuple[str, float]]:
    """
    Given a learner profile and completed course IDs, return top-k recommendations.
    """
    # Encode profile as embedding
    results = vectorstore.similarity_search_with_score(profile, k=20)
    
    # Filter out completed courses
    filtered = [(r.metadata["course_id"], score, r.page_content) 
                for r, score in results if r.metadata["course_id"] not in completed_ids]
    
    # Take top-k
    topk = sorted(filtered, key=lambda x: x[1])[:k]
    
    return [(cid, float(score)) for cid, score, _ in topk]

In [9]:
# Sample Input Queries

test_queries = [
    ("I’ve completed the 'Python Programming for Data Science' course and enjoy data visualization. What should I take next?", ["C101"]),
    ("I know Azure basics and want to manage containers and build CI/CD pipelines. Recommend courses.", ["C202"]),
    ("My background is in ML fundamentals; I’d like to specialize in neural networks and production workflows.", ["C303"]),
    ("I want to learn to build and deploy microservices with Kubernetes—what courses fit best?", ["C404"]),
    ("I’m interested in blockchain and smart contracts but have no prior experience. Which courses do you suggest?", []),
]

Create Evaluation and collect results

In [10]:
evaluation_results = []

for profile, completed in test_queries:
    recs = recommend_courses(profile, completed, k=5)
    for cid, score in recs:
        course_title = df[df["course_id"] == cid]["title"].values[0]
        evaluation_results.append({
            "profile": profile,
            "recommended_course_id": cid,
            "recommended_course_title": course_title,
            "similarity_score": score
        })

eval_df = pd.DataFrame(evaluation_results)
eval_df


,profile,recommended_course_id,recommended_course_title,similarity_score
0,I’ve completed the 'Python Programming for Dat...,C016,Python Programming for Data Science,0.481377
1,I’ve completed the 'Python Programming for Dat...,C017,R Programming and Statistical Analysis,0.958742
2,I’ve completed the 'Python Programming for Dat...,C014,Data Visualization with Tableau,1.080086
3,I’ve completed the 'Python Programming for Dat...,C011,Big Data Analytics with Spark,1.149961
4,I’ve completed the 'Python Programming for Dat...,C001,Foundations of Machine Learning,1.207464
5,I know Azure basics and want to manage contain...,C007,Cloud Computing with Azure,0.801474
6,I know Azure basics and want to manage contain...,C009,Containerization with Docker and Kubernetes,0.984642
7,I know Azure basics and want to manage contain...,C008,DevOps Practices and CI/CD,1.066410
8,I know Azure basics and want to manage contain...,C010,APIs and Microservices Architecture,1.213761
9,I know Azure basics and want to manage contain...,C006,Data Engineering on AWS,1.256097


Save evaluation results

In [11]:
eval_df.to_csv("assignment2_recommendations.csv", index=False)
print("Saved results to assignment2_recommendations.csv")


Saved results to assignment2_recommendations.csv
